# **Атрибуция для трансформеров: rollout против Chefer**

Практика к модулю [«Атрибуция от аксиом»](https://ai-interpretability.school).

Урок делает про attention rollout утверждение, которое можно проверить, а не принять на веру:
**карта не зависит от класса** — на снимке с собакой и кошкой rollout выдаст одно и то же,
о чем бы вы ни спрашивали. И второе, про Chefer et al.: классовую специфичность там
обеспечивает **градиент логита по матрице внимания**.

Проверим оба. Считает около минуты на процессоре.

**Про реализацию сразу и честно.** Ниже собран *generic*-вариант метода Chefer: матрицы
внимания взвешиваются градиентом, берется положительная часть, идет усреднение по головам
и прокрутка по слоям. Полный вариант из статьи добавляет к этому распространение
релевантности по правилам LRP — с отдельными правилами для skip-соединений и для произведения
$A\cdot V$. Он дает более чистые карты, но требует переписать весь проход по сети, и в тетрадь
не помещается. Классовую специфичность, ради которой все затевается, обеспечивает именно
градиентный множитель, и он здесь есть.

In [ ]:
!pip install timm -q     # в Colab timm бывает не установлен

In [ ]:
import io
import types
import urllib.request

import timm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'

model = timm.create_model('vit_small_patch16_224', pretrained=True).eval()
for p in model.parameters():
    p.requires_grad_(False)


def attn_forward(self, x, **kw):
    """Внимание с сохранением матрицы и градиента по ней: обоим методам нужны они."""
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    q, k = self.q_norm(q), self.k_norm(k)
    attn = ((q * self.scale) @ k.transpose(-2, -1)).softmax(dim=-1)
    self.attn_map = attn          # матрица нужна обоим методам
    attn.retain_grad()            # а градиент по ней — только Chefer
    return self.proj((attn @ v).transpose(1, 2).reshape(B, N, C))


for block in model.blocks:
    block.attn.fused_attn = False     # быстрое внимание матрицу не отдает, поэтому выключаем
    block.attn.forward = types.MethodType(attn_forward, block.attn)

cfg = model.pretrained_cfg
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(cfg['mean'], cfg['std'])])


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


img = tf(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB'))
img = img.unsqueeze(0).requires_grad_(True)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]
GRID = int(model.patch_embed.num_patches ** 0.5)
print(f'блоков: {len(model.blocks)}, сетка патчей: {GRID}x{GRID}')

## 1. Достать внимание и градиент по нему

Библиотека считает внимание быстрым ядром, которое саму матрицу наружу не отдает. Нам она
нужна — обоим методам, — поэтому быстрое внимание выключается и подставляется обычная
реализация в четыре строки.

Обратите внимание на `attn.retain_grad()`: без него градиент по промежуточному тензору
не сохранится, и Chefer посчитать будет нечем.

In [ ]:
def run(cls=None):
    """Один прогон: матрицы внимания всех блоков и градиенты логита класса по ним."""
    model.zero_grad()
    img.grad = None
    out = model(img)
    c = int(out.argmax()) if cls is None else cls
    out[0, c].backward()
    return c, [b.attn.attn_map.detach()[0] for b in model.blocks], \
        [b.attn.attn_map.grad[0] for b in model.blocks]


predicted, mats, grads = run()
print(f'спрогнозированный класс: {predicted} {names[predicted]}')
print(f'форма матрицы внимания (головы, токены, токены): {tuple(mats[0].shape)}')

## 2. Два метода

Разница между ними — в одной строке, и стоит посмотреть, в какой именно.

**rollout** усредняет матрицу по головам и добавляет единичную — это и есть учет
skip-соединений, о котором говорит урок: половина веса отдается вниманию, половина прямому
проносу.

**Chefer** делает то же самое, но перед усреднением домножает матрицу на градиент логита
по ней и берет положительную часть. **Класс попадает в метод ровно здесь** — больше нигде
в этих двух функциях он не упоминается.

In [ ]:
def rollout(mats):
    """Attention rollout: усреднение по головам, добавление единичной, произведение по слоям."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A in mats:
        a = 0.5 * A.mean(0) + 0.5 * torch.eye(n)     # единичная матрица — тот самый учет skip-соединений
        r = a @ r
    return r


def chefer(mats, grads):
    """Chefer et al.: та же прокрутка, но каждая матрица взвешена градиентом логита класса."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A, G in zip(mats, grads):
        a = (G * A).clamp(min=0).mean(0)             # вот здесь и появляется класс — через градиент
        a = a + torch.eye(n)
        a = a / a.sum(dim=-1, keepdim=True)
        r = a @ r
    return r


def to_map(r):
    """Строка токена CLS без него самого, разложенная обратно в сетку патчей."""
    m = r[0, 1:].reshape(GRID, GRID)
    return (m - m.min()) / (m.max() - m.min() + 1e-9)

## 3. Главная проверка: зависит ли карта от вопроса

Строим по две карты каждым методом — спрашивая про кошку и про собаку — и сравниваем.

In [ ]:
CAT, DOG = 281, 243        # tabby и bull mastiff — кошка и собака
maps = {}
for cls, label in ((CAT, f'кошка'), (DOG, f'собака')):
    _, m, g = run(cls)
    maps[label] = (to_map(rollout(m)), to_map(chefer(m, g)))


def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm() + 1e-9))


for i, name in ((0, 'rollout'), (1, 'Chefer')):
    a, b = maps[f'кошка'][i], maps[f'собака'][i]
    print(f'   {name:9} корреляция карт кошки и собаки: {corr(a, b):.6f}   максимальное расхождение: {(a - b).abs().max():.2e}')

**У rollout корреляция ровно единица, а расхождение ровно ноль.** Не «карты
похожи» — они побитово одинаковые. Утверждение урока оказалось не фигурой речи: матрица
внимания считается один раз при прямом проходе и про класс не знает ничего, значит и
произведение матриц про класс не знает.

**У Chefer корреляция около $0{,}7$**, а расхождение — больше половины диапазона карты. Карты
разные, и разными их сделал один множитель.

**Задание 1.** Возьмите третий класс — что-нибудь заведомо неподходящее, например `849`
(чайник). Останется ли карта rollout прежней? А карта Chefer — насколько она отличается
от карты для собаки?

In [ ]:
# Ваш код здесь

## 4. Куда именно смотрят карты

Корреляция говорит, что карты разные, но не говорит, **правильно** ли они разные. Разметим
рамки кошки и собаки и посмотрим, какая доля массы карты попадает в каждую.

In [ ]:
# Рамки размечены на глаз по снимку 224 на 224
BOXES = {f'кошка': (10, 40, 110, 200), f'собака': (115, 30, 215, 205)}


def mass_in(m, box):
    """Доля массы карты, попавшая внутрь рамки."""
    big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
    x0, y0, x1, y1 = box
    return float(big[y0:y1, x0:x1].sum() / big.sum())


print(f'{"":10}{"спросили":12}{"в рамке кошки":>16}{"в рамке собаки":>17}')
for label in (f'кошка', f'собака'):
    for i, name in ((0, 'rollout'), (1, 'Chefer')):
        m = maps[label][i]
        print(f'{name:10}{label:12}{mass_in(m, BOXES[f"кошка"]) * 100:15.1f}%'
              f'{mass_in(m, BOXES[f"собака"]) * 100:16.1f}%')

Ради этой таблицы все и затевалось.

**Строки rollout совпадают до десятой доли процента** — и не потому, что метод плох, а потому
что он отвечает на вопрос «куда вообще смотрит модель», а не «почему именно этот класс».
Как Eigen-CAM из модуля про CAM-семейство.

**Строки Chefer расходятся, и расходятся в нужную сторону:** спросили про кошку — больше массы
легло в рамку кошки; спросили про собаку — перевес ушел в рамку собаки. Это и есть классовая
специфичность, померенная числом.

**Задание 2.** Постройте ту же таблицу для класса, которого на снимке нет вовсе. Куда ляжет
масса у Chefer? О чем это говорит: метод нашел несуществующий объект или честно показал, за
что модель зацепилась бы, если бы ее заставили отвечать про этот класс?

In [ ]:
# Ваш код здесь

## 5. Посмотреть глазами

Числа сказали главное, но карты стоит и увидеть.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, label in enumerate((f'кошка', f'собака')):
    for col, name in enumerate(('rollout', 'Chefer')):
        axes[row, col].imshow(maps[label][col].numpy(), cmap='inferno')
        axes[row, col].set_title(f'{name} — {label}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

**Задание 3.** Урок называет третий вариант для трансформеров — Grad-CAM на
переставленных токенах, из модуля про CAM-семейство. Постройте его на том же снимке и для тех
же двух классов и добавьте строку в таблицу с рамками. Где он окажется между rollout
и Chefer — ближе к классово-слепому или к классово-специфичному?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Attention rollout классово слеп, и это проверяется побитово.** Корреляция карт для двух
  разных классов равна единице, расхождение — нулю. Если вам нужен ответ на вопрос «почему
  именно этот класс», rollout его не даст никогда, сколько ни улучшай реализацию.
- **Класс входит в Chefer ровно через градиент**, и это видно в коде: одна строка, в которой
  матрица внимания домножается на $\partial y^c/\partial A$. Уберите ее — получите rollout.
- **Корреляции мало, нужна рамка.** «Карты разные» — еще не «карты правильные». Доля массы
  внутри размеченной области отвечает на второй вопрос, и отвечает числом.
- **Полный Chefer сложнее собранного здесь**: там добавляются правила LRP для skip-соединений
  и для произведения $A\cdot V$. Карты выходят чище, но классовую специфичность дает не они,
  а градиентный множитель — тот, что здесь уже есть.